# 06 · matplotlib 制图（基础）

用 matplotlib 把数据与几何形状画成静态图，掌握绘图接口、常用图型、形状绘制、图面标注与中文字体设置。

## 学习目标
- 分辨状态式 state-based（plt.xxx）与对象式 object-oriented（fig, ax）两种绘图接口，并能在对象式接口上作图。
- 控制图的尺寸与分辨率（figsize / dpi），并正确使用 show / savefig / close / tight_layout 管理输出与资源。
- 选用合适的常用图型：散点图 scatter、折线图 plot、长条图 bar、直方图 histogram。
- 用 patches 与 collections 绘制几何多边形，并以等比例座标 set_aspect equal 维持形状正确。
- 为图面加上完整标注：标题、轴标签、图例 legend、文字 text、格线 grid 与轴的开关。
- 设置中文字体 font，让图中中文与负号正常显示。

In [ ]:
# 中文字体设置（本书会画图才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 两种绘图接口与图对象

matplotlib 有两种作图方式。状态式接口 state-based interface 用 `plt.plot`、`plt.title` 等函数，对「目前这张图」连续下指令，适合快速试画。对象式接口 object-oriented interface 则先取得 figure 与 axes 两个对象，再对它们调用方法。

心智模型：figure 是整块画布，axes 是画布上的一个座标系（一张子图）。`fig, ax = plt.subplots()` 一次拿到两者。

为什么学对象式：一张画布上有多个座标系时，状态式难以指明对象；对象式可控、可重用，正式图较不易出错。本书后续一律以对象式为主。

形状：
```
fig, ax = plt.subplots()
ax.plot(x, y)
```

In [ ]:
# 概念：同一条折线分别用状态式与对象式画出，对照两种写法
import numpy as np

# 造一小串数列当示范数据
x = np.arange(0, 6)                 # 0 到 5 共 6 个整数
y = np.array([1, 3, 2, 5, 4, 6])    # 对应的 y 值（自造）

# 状态式：对「目前的图」连续下指令
plt.plot(x, y)                      # 在当前 figure 上画线
plt.title('状态式接口')             # 设置当前图的标题
plt.show()                          # 显示并结束这张图

# 对象式：先取得 fig 与 ax，再对 ax 下指令
fig, ax = plt.subplots()            # fig 是画布，ax 是座标系
ax.plot(x, y)                       # 在指定的 ax 上画线
ax.set_title('对象式接口')          # 对该 ax 设置标题（注意：方法名是 set_title）
plt.show()

print('fig 类型：', type(fig).__name__)
print('ax  类型：', type(ax).__name__)

## 图的尺寸、输出与资源管理

图的实际像素由两个参数共同决定：图尺寸 figsize（单位英吋，宽高）与分辨率 dpi（每英吋像素数）。输出像素约为 figsize × dpi，dpi 越高越清晰、文件越大。

输出与资源相关方法：

| 方法 | 作用 |
|---|---|
| show | 在画面上显示图 |
| savefig | 把图存成文件（可指定 dpi） |
| close | 关闭图、释放内存 |
| tight_layout | 自动调整边距，避免标签被裁切 |

为什么学：批量作图时若不 `close`，图对象会累积占用内存；`tight_layout` 则避免轴标签或标题被切掉。

In [ ]:
# 概念：同一张图以不同 figsize 与 dpi 各存一份，比较尺寸后关闭释放
x = np.linspace(0, 10, 100)         # 0 到 10 取 100 个等距点
y = np.sin(x)                       # 对每点取 sin（矢量化，无需循环）

settings = [(4, 3, 80), (8, 6, 150)]   # 各组 (宽英吋, 高英吋, dpi)
for w, h, d in settings:
    fig, ax = plt.subplots(figsize=(w, h), dpi=d)   # figsize 与 dpi 共同决定像素
    ax.plot(x, y)
    ax.set_title('正弦曲线')
    fig.tight_layout()              # 自动排版，避免标签被裁切
    fname = f'sine_{w}x{h}_{d}dpi.png'
    fig.savefig(fname)              # 存盘，沿用该图的 dpi
    # 注意：批量作图务必 close，否则图对象累积占内存
    plt.close(fig)                  # 关闭并释放这张图
    px_w, px_h = w * d, h * d       # 估算输出像素
    print(f'{fname}: 约 {px_w} x {px_h} 像素')

print('已存盘的图档：', [f for f in os.listdir('.') if f.startswith('sine_')])

## 常用图型

选图原则：数据长什么样，就配什么图。

| 图型 | 方法 | 适用 | 常用参数 |
|---|---|---|---|
| 折线图 plot | ax.plot | 看趋势、有序变化 | color、linestyle |
| 散点图 scatter | ax.scatter | 看两变量分布与关系 | color、marker、s |
| 长条图 bar | ax.bar | 比较类别数值 | color、width |
| 直方图 histogram | ax.hist | 看单一数值的分布 | bins、color |

判断力：折线强调趋势、散点看分布、长条比类别、直方看数值分布。

In [ ]:
# 概念：在 2x2 子座标各画一种图型并列对照
rng = np.random.default_rng(0)      # 固定乱数种子，结果可重现

# 自造示范数据
t = np.arange(12)                   # 12 个时间点
trend = np.cumsum(rng.normal(0, 1, 12))   # 累积和仿真一条趋势
px = rng.normal(0, 1, 50)           # 散点的 x（50 点）
py = px * 0.8 + rng.normal(0, 0.5, 50)    # 与 px 相关的 y
cats = ['A', 'B', 'C', 'D']         # 四个类别名称
vals = [5, 9, 4, 7]                 # 各类别数值
samples = rng.normal(50, 10, 300)   # 300 笔数值，看其分布

fig, axes = plt.subplots(2, 2, figsize=(9, 7))   # 2x2 共四个座标系
axes[0, 0].plot(t, trend)                        # 折线：看趋势
axes[0, 0].set_title('折线图 plot')
axes[0, 1].scatter(px, py, s=15)                 # 散点：看分布（s 是点大小）
axes[0, 1].set_title('散点图 scatter')
axes[1, 0].bar(cats, vals)                       # 长条：比类别
axes[1, 0].set_title('长条图 bar')
axes[1, 1].hist(samples, bins=20)                # 直方：看数值分布（bins 是分组数）
axes[1, 1].set_title('直方图 histogram')
fig.tight_layout()
plt.show()
plt.close(fig)

## 几何多边形绘制

matplotlib 不只画数据，也能画几何图形。常用组件：

| 组件 | 用途 |
|---|---|
| patches.Rectangle | 矩形（左下角座标 + 宽高） |
| patches.Polygon | 多边形（一串顶点座标） |
| collections.LineCollection | 一次画多条线段的集合 |

把形状加入座标系要用 `ax.add_patch`（单一形状）或 `ax.add_collection`（线段集合）。

关键：`ax.set_aspect('equal')` 让 x、y 轴等比例，正方形、圆形与角度才不会被拉扁变形。

In [ ]:
# 概念：自造一个多边形与几条线段，画出后设等比例座标确认不变形
from matplotlib import patches
from matplotlib.collections import LineCollection

# 自造一个五边形的顶点座标
poly_pts = np.array([[1, 1], [3, 1], [4, 3], [2, 4], [0, 3]])

# 自造几条线段，每条是 [(起点), (终点)]
segments = [[(0, 0), (5, 0)], [(5, 0), (5, 5)], [(0, 0), (0, 5)]]

fig, ax = plt.subplots(figsize=(5, 5))
rect = patches.Rectangle((0.5, 0.5), 1, 1, facecolor='lightgray')   # 左下角(0.5,0.5)、宽高各1
ax.add_patch(rect)                                   # 把矩形加入座标系
polygon = patches.Polygon(poly_pts, closed=True, facecolor='skyblue', edgecolor='navy')
ax.add_patch(polygon)                                # 把多边形加入座标系
lines = LineCollection(segments, colors='red', linewidths=2)
ax.add_collection(lines)                             # 把线段集合加入座标系
ax.set_xlim(-1, 6)                                   # add_patch 不会自动缩放，需手动设范围
ax.set_ylim(-1, 6)
# 技巧：set_aspect('equal') 是维持形状正确的关键
ax.set_aspect('equal')                               # x、y 同比例，形状不被拉扁
ax.set_title('几何多边形与线段')
plt.show()
plt.close(fig)
print('多边形顶点数：', len(poly_pts))

## 图面标注与座标控制

标注让图能被独立读懂，不必再口头解释。常用标注：

| 标注 | 方法 | 说明 |
|---|---|---|
| 标题 title | ax.set_title | 图的主题 |
| 轴标签 xlabel/ylabel | ax.set_xlabel/set_ylabel | 说明各轴代表什么 |
| 图例 legend | ax.legend | 依各线的 label 自动生成 |
| 文字 text | ax.text | 在指定数据座标放文字 |
| 格线 grid | ax.grid | 辅助对齐读值 |

重点：`legend` 依赖画图时给的 `label`；`text` 以数据座标定位；纯几何图常用 `ax.axis('off')` 关闭座标轴。

In [ ]:
# 概念：取一张折线图补上完整标注
x = np.linspace(0, 10, 50)
y1 = np.sin(x)
y2 = np.cos(x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y1, label='sin')         # label 供 legend 使用
ax.plot(x, y2, label='cos')
ax.set_title('正弦与余弦')          # 标题
ax.set_xlabel('x 值')               # x 轴标签
ax.set_ylabel('函数值')             # y 轴标签
ax.legend()                         # 依 label 生成图例
ax.grid(True)                       # 开格线辅助读值
# 技巧：text 用数据座标定位，这里标在 (x=5, y=1) 附近
ax.text(5, 1.0, '交会区', color='purple')
plt.show()
plt.close(fig)

## 中文字体设置

matplotlib 预设字体不含中文字符，直接画中文会变成方框（俗称豆腐字）；而预设的负号 minus 也可能在某些字体下显示为乱码。

解法两步：
1. 用字体管理 font_manager 的 `addfont` 注册一个含中文的字库，再透过全域设置 `rcParams['font.family']` 指定字体族。
2. 设 `rcParams['axes.unicode_minus'] = False`，改用一般 ASCII 减号，负号才不会乱码。

本书第二个 cell 已做过全域设置，此处示范验证设置后中文与负号都正常。

In [ ]:
# 概念：画一张含中文标题与负值座标的图，确认字体设置有效
# 注意：字体族与 unicode_minus 已在本书第二个 cell 全域设置，这里直接沿用
x = np.linspace(-5, 5, 100)         # 含负值的座标范围
y = x ** 3                          # 立方，y 也会出现负值

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, y)
ax.set_title('三次方曲线（含中文与负号）')   # 中文标题应正常显示
ax.set_xlabel('输入值')
ax.set_ylabel('输出值')
ax.axhline(0, color='gray', linewidth=0.8)   # 画 y=0 水平线，凸显负区
plt.show()
plt.close(fig)

print('目前字体族设置：', plt.rcParams['font.family'])
print('unicode_minus：', plt.rcParams['axes.unicode_minus'])

## 练习

以下三题由浅到深，皆为复合型但简短。数据一律自己用 numpy 或 list 造，不读外部档。

In [ ]:
# TODO 1 ·（简单）建筑楼高分布折线图（集成：对象式接口 object-oriented interface + 折线图 plot）
#   情境：用一串自造的建筑楼高 height（公尺）数据，画随建筑编号变化的楼高折线图。
#   要求：
#     1. 用 numpy 或 list 自造 10 栋建筑的楼高数值（如 12、24、36… 公尺）。
#     2. 以对象式接口 fig, ax = plt.subplots() 创建一张图。
#     3. 在 ax 上用 ax.plot 画楼高折线，并设中文标题与轴标签（x 轴：建筑编号，y 轴：楼高（公尺））。
#   验收：应看到一条由左到右、随建筑编号起伏的楼高折线图，标题与轴标签皆为中文。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）街廓建筑概况四格图（集成：对象式接口 + 常用图型 + 图面标注 + 中文字体 font）
#   情境：把一个街廓 block 内多栋建筑的指针整理成一张四格对照图并存盘。
#   要求：
#     1. 用 numpy 自造同一批建筑的楼高 height、楼层数 floors、占地面积 footprint area、容积率 FAR 等数列。
#     2. 以对象式接口建 2x2 子座标，四格分别画折线图 plot、长条图 bar、散点图 scatter、直方图 histogram
#        （例如散点看楼高与楼层数关系、直方看 FAR 分布）。
#     3. 为每格加上中文标题与轴标签（字体已于本书第二个 cell 设置）。
#     4. 用 tight_layout 排版后以指定 dpi savefig 存盘，最后 close 释放资源。
#   验收：应看到一张 2x2、四种图型并列、中文标注正常的街廓概况图档，且程序结束未残留未关闭的图。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）政策前后容积比较与街廓示意图
#   （集成：对象式接口 + 常用图型 + 几何多边形绘制 + 座标控制 set_aspect equal + 图面标注 + 中文字体 font）
#   情境：比较同一街廓在容积率政策调整（套用一个高度乘数）前后的楼地板面积 GFA，并画街廓内置筑平面示意。
#   要求：
#     1. 用 numpy 自造每栋建筑的占地面积 footprint area 与容积率 FAR，算出现况 GFA；
#        再对 FAR 套一个高度乘数（如 1.2）得到政策后 GFA。
#     2. 左图以长条图 bar 并列比较各栋政策前后 GFA，并用 ax.text 标出总 GFA 增幅（聚合 aggregation 后相加比较）。
#     3. 右图用 patches.Rectangle 依各栋占地面积在网格 cell 排平面方块，
#        以 collections.LineCollection 画街廓边界，并用 set_aspect('equal') 维持比例、axis off 关闭座标轴。
#     4. 全图中文字体与负号正常，加上总标题与图例 legend。
#   验收：左侧为政策前后 GFA 并列长条与标出的总增幅文字；右侧为等比例、不变形且关闭座标轴的街廓平面示意图。
# TODO: 在这里完成


## 小结

- matplotlib 有状态式与对象式两种接口；对象式以 `fig, ax = plt.subplots()` 取得画布与座标系，可控可重用，是正式作图首选。
- 输出像素由 figsize 与 dpi 共同决定；批量作图要用 `close` 释放内存、`tight_layout` 避免标签被裁切。
- 依数据形态选图：折线看趋势、散点看分布、长条比类别、直方看数值分布。
- patches 与 collections 能画几何形状，`set_aspect('equal')` 是维持形状不变形的关键。
- 标题、轴标签、legend、text、grid 让图能被独立读懂；legend 依赖 label，text 用数据座标定位。
- 用 font_manager 注册字体并透过 rcParams 设为全域，再关闭 unicode_minus，中文与负号才能正常显示。